In [1]:
!pip install pygit2==1.15.1
%cd /content
!git clone https://github.com/lllyasviel/Fooocus.git
%cd /content/Fooocus
!python entry_with_update.py --share --always-high-vram


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 768.8/768.8 kB 45.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for pygit2 (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for pygit2
Failed to build pygit2
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (pygit2)
/content
Cloning into 'Fooocus'...
remote: Enumerating objects: 6735, done.
remote: Total 6735 (delta 0), reused 0 (delta 0), pack-reused 6735 (from 1)
Receiving objects: 100% (6735/6735), 33.35 MiB | 20.42 MiB/s, done.
Resolving deltas: 100% (3849/3849), done.
/content/Fooocus
Already up-to-date
Update succeeded.
[System ARGV] 

In [2]:
import pandas as pd

In [7]:
import pandas as pd

try:
    excel_data = pd.read_excel('/content/dados.xlsx', sheet_name=None)

    print("Arquivo 'dados.xlsx' carregado com sucesso!")
    print("Abas encontradas:")
    for sheet_name, df in excel_data.items():
        print(f"  - {sheet_name} (linhas: {len(df)}, colunas: {len(df.columns)})")

except FileNotFoundError:
    print("Erro: O arquivo 'dados.xlsx' não foi encontrado. Por favor, certifique-se de que o arquivo está no diretório correto.")
except Exception as e:
    print(f"Ocorreu um erro ao carregar o arquivo Excel: {e}")

Arquivo 'dados.xlsx' carregado com sucesso!
Abas encontradas:
  - 10 (linhas: 56687, colunas: 44)
  - 558 (linhas: 354, colunas: 44)
  - 1936 (linhas: 31, colunas: 44)


In [12]:
import pandas as pd

print("Valores únicos da coluna 'Matrícula' em cada aba (primeiros 10):")
for sheet_name, df in excel_data.items():
    if 'Matrícula' in df.columns:
        unique_matriculas = df['Matrícula'].astype(str).unique()
        if len(unique_matriculas) > 10:
            print(f"  Aba '{sheet_name}': {unique_matriculas[:10].tolist()} ... (e mais {len(unique_matriculas) - 10} valores)")
        else:
            print(f"  Aba '{sheet_name}': {unique_matriculas.tolist()}")
    else:
        print(f"  Aba '{sheet_name}': Coluna 'Matrícula' não encontrada.")

# O código para filtrar o DataFrame será adicionado aqui após identificar a coluna e o valor correto.

Valores únicos da coluna 'Matrícula' em cada aba (primeiros 10):
  Aba '10': ['131772.0000314.0', '004653.0010624.0', '435788.0000001.0', '000000.0728674.0', '000000.0752536.0', '131894.0001833.0', '000000.0737560.0', '096486.0001659.0', '004653.0091179.0', '004653.0103483.0'] ... (e mais 36141 valores)
  Aba '558': ['004653.0090209.0', '004653.0117145.0', '004653.0063897.0', '004653.0131467.0', '004653.0056914.0', '004653.0080594.0', '004653.0035369.0', '000995.0000024.0', '004653.0093232.0', '004653.0099800.0'] ... (e mais 333 valores)
  Aba '1936': ['004653.0047248.0', '004653.0141523.0', '004653.0030431.0', '004653.0052309.0', '004653.0133685.0', '004653.0014380.0', '004653.0037423.0', '004653.0063358.0', '004653.0141860.0', '004653.0055411.0'] ... (e mais 21 valores)


In [13]:
monthly_counts_all_sheets = pd.DataFrame()

print("Processando 'Data de entrada' para cada aba:")
for sheet_name, df in excel_data.items():
    if 'Data de entrada' in df.columns:
        # Converte a coluna 'Data de entrada' para datetime, tratando erros
        df['Data de entrada'] = pd.to_datetime(df['Data de entrada'], errors='coerce')

        # Remove linhas onde a conversão falhou (valores NaT)
        df_clean = df.dropna(subset=['Data de entrada'])

        # Extrai o ano e o mês
        df_clean['AnoMes'] = df_clean['Data de entrada'].dt.to_period('M')

        # Conta a quantidade por mês
        monthly_counts_sheet = df_clean['AnoMes'].value_counts().sort_index().reset_index()
        monthly_counts_sheet.columns = ['AnoMes', 'Quantidade']
        monthly_counts_sheet['Aba'] = sheet_name

        print(f"  - Aba '{sheet_name}':\n{monthly_counts_sheet.to_string(index=False)}\n")
        monthly_counts_all_sheets = pd.concat([monthly_counts_all_sheets, monthly_counts_sheet])
    else:
        print(f"  - Aba '{sheet_name}': Coluna 'Data de entrada' não encontrada.")

if not monthly_counts_all_sheets.empty:
    print("\n--- Quantidade total por mês (todas as abas) ---")
    total_monthly_counts = monthly_counts_all_sheets.groupby('AnoMes')['Quantidade'].sum().reset_index()
    total_monthly_counts = total_monthly_counts.sort_values(by='AnoMes').reset_index(drop=True)
    print(total_monthly_counts.to_string(index=False))
else:
    print("\nNenhum dado com 'Data de entrada' válida foi encontrado para análise mensal.")


Processando 'Data de entrada' para cada aba:
  - Aba '10':
 AnoMes  Quantidade Aba
2026-07       22582  10
2026-08       22467  10
2026-09       11638  10

  - Aba '558':
 AnoMes  Quantidade Aba
2026-07         129 558
2026-08         168 558
2026-09          57 558

  - Aba '1936':
 AnoMes  Quantidade  Aba
2026-07          14 1936
2026-08          17 1936


--- Quantidade total por mês (todas as abas) ---
 AnoMes  Quantidade
2026-07       22725
2026-08       22652
2026-09       11695


Com base na sua confirmação, vamos agora filtrar os dados para encontrar todas as entradas onde a coluna 'Matrícula' começa com '004653'. Farei isso em todas as abas e concatenarei os resultados em um único DataFrame.

In [14]:
filtered_data_004653 = pd.DataFrame()
matricula_prefix = '004653'

print(f"Filtrando matrículas que começam com '{matricula_prefix}' em cada aba:")
for sheet_name, df in excel_data.items():
    if 'Matrícula' in df.columns:
        # Converte para string para garantir que o método .startswith() funcione
        df_filtered_sheet = df[df['Matrícula'].astype(str).str.startswith(matricula_prefix, na=False)]

        if not df_filtered_sheet.empty:
            print(f"  - Aba '{sheet_name}': Encontrado {len(df_filtered_sheet)} registros.")
            filtered_data_004653 = pd.concat([filtered_data_004653, df_filtered_sheet], ignore_index=True)
        else:
            print(f"  - Aba '{sheet_name}': Nenhum registro encontrado com matrícula começando com '{matricula_prefix}'.")
    else:
        print(f"  - Aba '{sheet_name}': Coluna 'Matrícula' não encontrada.")

print(f"\nTotal de registros encontrados em todas as abas com matrícula começando com '{matricula_prefix}': {len(filtered_data_004653)}.")

if not filtered_data_004653.empty:
    print("Primeiras 5 linhas do DataFrame filtrado:")
    display(filtered_data_004653.head())
else:
    print("Nenhum registro foi encontrado após o filtro.")

Filtrando matrículas que começam com '004653' em cada aba:
  - Aba '10': Encontrado 14452 registros.
  - Aba '558': Encontrado 348 registros.
  - Aba '1936': Encontrado 31 registros.

Total de registros encontrados em todas as abas com matrícula começando com '004653': 14831.
Primeiras 5 linhas do DataFrame filtrado:


,Tipo Atendimento,No.Atendimento,Operador,Data de entrada,Matrícula,Nome,Plano,Telefone,Endereço,Bairro,...,Idade,motivo,subMotivo,TipoPlano,Possui anexo,Urgencia,Empresa,Nome da Empresa,Data de inicio do Contrato,Situação Contrato
0,10-INFORMACAO_CONTRATUAL,30922220260701002112,AABREU,2026-07-01 05:30:47,004653.0010624.0,FABIO DA SILVA FERREIRA,737CP - RIOSERV MAX QC,21 970073004/(21) 3272-8734,"RUA AMALIA 108 FUNDOS, CASA 09",QUINTINO BOCAIUVA,...,49,NaN,NaN,BASICO,NaN,NaN,4653,PREFEITURA DA CIDADE DO RIO DE JANEIRO,42156.0,Ativo
1,10-INFORMACAO_CONTRATUAL,30922220260701003917,SIMONEF,2026-07-01 07:15:18,004653.0091179.0,ANDREA SAMPAIO SANT ANNA,735CP - RIOSERV INTERMEDIARIO QP,(21) 3111-4130,RUA ARISTIDES CAIRE 79 APT. 701,MEIER,...,57,NaN,NaN,INTERMEDIARIO,NaN,NaN,4653,PREFEITURA DA CIDADE DO RIO DE JANEIRO,42156.0,Ativo
2,10-INFORMACAO_CONTRATUAL,30922220260701003959,RAFFAELP,2026-07-01 07:12:57,004653.0103483.0,LOURDES DE FATIMA FERREIRA D ALMEIDA,737CP - RIOSERV MAX QC,(21)7878-1561,TRAVESSA ENEIDA 81 APARTAMENTO 201,PORTUGUESA,...,68,NaN,NaN,BASICO,NaN,NaN,4653,PREFEITURA DA CIDADE DO RIO DE JANEIRO,42156.0,Ativo
3,10-INFORMACAO_CONTRATUAL,30922220260701006404,ANAPS,2026-07-01 07:50:24,004653.0041301.0,FATIMA MARIA RODRIGUES DO NASCIMENTO,737CP - RIOSERV MAX QC,(21)3217-2082,ESTRADA DO MENDANHA 2870 BL 01 APT 402,CAMPO GRANDE,...,65,NaN,NaN,BASICO,NaN,NaN,4653,PREFEITURA DA CIDADE DO RIO DE JANEIRO,42156.0,Ativo
4,10-INFORMACAO_CONTRATUAL,30922220260701006939,DRIALVES,2026-07-01 08:01:57,004653.0030161.0,AVANI MARTINS DE FRANCA,737CP - RIOSERV MAX QC,(21)0000-0000,RUA GETULIO 321 APART 1007 BLOCO 2,TODOS OS SANTOS,...,71,NaN,NaN,BASICO,NaN,NaN,4653,PREFEITURA DA CIDADE DO RIO DE JANEIRO,42156.0,Ativo


Agora, vamos calcular a quantidade de entradas por mês para o DataFrame `filtered_data_004653`.

In [15]:
if not filtered_data_004653.empty:
    # Garante que 'Data de entrada' é do tipo datetime
    filtered_data_004653['Data de entrada'] = pd.to_datetime(filtered_data_004653['Data de entrada'], errors='coerce')

    # Remove linhas com 'Data de entrada' inválida após a conversão
    df_filtered_clean = filtered_data_004653.dropna(subset=['Data de entrada'])

    if not df_filtered_clean.empty:
        # Extrai o ano e o mês
        df_filtered_clean['AnoMes'] = df_filtered_clean['Data de entrada'].dt.to_period('M')

        # Conta a quantidade por mês para o DataFrame filtrado
        monthly_counts_filtered = df_filtered_clean['AnoMes'].value_counts().sort_index().reset_index()
        monthly_counts_filtered.columns = ['AnoMes', 'Quantidade']

        print("\n--- Quantidade de entradas por mês para matrículas começando com '004653' ---")
        print(monthly_counts_filtered.to_string(index=False))
    else:
        print("Nenhuma 'Data de entrada' válida encontrada no DataFrame filtrado para análise mensal.")
else:
    print("O DataFrame filtrado está vazio. Não é possível calcular a quantidade por mês.")


--- Quantidade de entradas por mês para matrículas começando com '004653' ---
 AnoMes  Quantidade
2026-07        5629
2026-08        6343
2026-09        2859


Para exibir as colunas por aba de origem, vamos recriar o DataFrame filtrado `filtered_data_004653` e adicionar uma coluna 'Aba Original' para identificar de qual aba cada registro veio.

In [16]:
filtered_data_with_sheets = pd.DataFrame()
matricula_prefix = '004653'

print(f"Recriando DataFrame filtrado com informação da aba para matrículas começando com '{matricula_prefix}':")
for sheet_name, df in excel_data.items():
    if 'Matrícula' in df.columns:
        # Converte para string para garantir que o método .startswith() funcione
        df_filtered_sheet = df[df['Matrícula'].astype(str).str.startswith(matricula_prefix, na=False)]

        if not df_filtered_sheet.empty:
            # Adiciona uma coluna 'Aba Original' antes de concatenar
            df_filtered_sheet['Aba Original'] = sheet_name
            filtered_data_with_sheets = pd.concat([filtered_data_with_sheets, df_filtered_sheet], ignore_index=True)
            print(f"  - Aba '{sheet_name}': {len(df_filtered_sheet)} registros adicionados.")
        else:
            print(f"  - Aba '{sheet_name}': Nenhum registro encontrado com matrícula começando com '{matricula_prefix}'.")
    else:
        print(f"  - Aba '{sheet_name}': Coluna 'Matrícula' não encontrada.")

print(f"\nTotal de registros no novo DataFrame filtrado: {len(filtered_data_with_sheets)}.")

if not filtered_data_with_sheets.empty:
    print("\n--- Colunas para cada Aba Original no DataFrame filtrado ---")
    for original_sheet in filtered_data_with_sheets['Aba Original'].unique():
        df_subset = filtered_data_with_sheets[filtered_data_with_sheets['Aba Original'] == original_sheet]
        print(f"Aba Original '{original_sheet}':")
        print(df_subset.columns.tolist())
        print("\n")
else:
    print("O DataFrame filtrado está vazio. Não é possível mostrar as colunas por aba.")

Recriando DataFrame filtrado com informação da aba para matrículas começando com '004653':
  - Aba '10': 14452 registros adicionados.
  - Aba '558': 348 registros adicionados.
  - Aba '1936': 31 registros adicionados.

Total de registros no novo DataFrame filtrado: 14831.

--- Colunas para cada Aba Original no DataFrame filtrado ---
Aba Original '10':
['Tipo Atendimento', 'No.Atendimento', 'Operador', 'Data de entrada', 'Matrícula', 'Nome', 'Plano', 'Telefone', 'Endereço', 'Bairro', 'Cidade', 'Status', 'Data de fechamento', 'Motivo', 'Ocorrência', 'Providências', 'Pjotinha', 'Senha', 'Código do Solicitante', 'Solicitante', 'CRM', 'Código do Grupo', 'Grupo', 'Especialidade', 'Código do Executor', 'Executor', 'Código do Grupo.1', 'Grupo.1', 'CID', 'Diagnóstico', 'TUSS Inicial', 'Operador Final', 'PRC', 'Tempo de Assim', 'Idade', 'motivo', 'subMotivo', 'TipoPlano', 'Possui anexo', 'Urgencia', 'Empresa', 'Nome da Empresa', 'Data de inicio do Contrato', 'Situação Contrato', 'Aba Original']


/tmp/ipykernel_1214/4005141158.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered_sheet['Aba Original'] = sheet_name
/tmp/ipykernel_1214/4005141158.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered_sheet['Aba Original'] = sheet_name


Vamos agora calcular a quantidade de entradas por mês para o DataFrame `filtered_data_with_sheets`, separando por cada 'Aba Original'.

In [17]:
if not filtered_data_with_sheets.empty:
    # Garante que 'Data de entrada' é do tipo datetime
    filtered_data_with_sheets['Data de entrada'] = pd.to_datetime(filtered_data_with_sheets['Data de entrada'], errors='coerce')

    # Remove linhas com 'Data de entrada' inválida após a conversão
    df_filtered_cleaned_with_sheets = filtered_data_with_sheets.dropna(subset=['Data de entrada'])

    if not df_filtered_cleaned_with_sheets.empty:
        # Extrai o ano e o mês
        df_filtered_cleaned_with_sheets['AnoMes'] = df_filtered_cleaned_with_sheets['Data de entrada'].dt.to_period('M')

        # Conta a quantidade por mês para cada 'Aba Original'
        monthly_counts_by_sheet = df_filtered_cleaned_with_sheets.groupby(['AnoMes', 'Aba Original']).size().unstack(fill_value=0)

        # Calcula o total mensal
        monthly_counts_by_sheet['Total'] = monthly_counts_by_sheet.sum(axis=1)

        print("\n--- Quantidade de entradas por mês por Aba Original (Matrículas começando com '004653') ---")
        print(monthly_counts_by_sheet.to_string())
    else:
        print("Nenhuma 'Data de entrada' válida encontrada no DataFrame filtrado por aba para análise mensal.")
else:
    print("O DataFrame filtrado por aba está vazio. Não é possível calcular a quantidade por mês.")


--- Quantidade de entradas por mês por Aba Original (Matrículas começando com '004653') ---
Aba Original    10  1936  558  Total
AnoMes                              
2026-07       5490    14  125   5629
2026-08       6159    17  167   6343
2026-09       2803     0   56   2859


### Filtrar a coluna 'Ocorrência' por palavras-chave

Agora, vamos filtrar os dados para encontrar registros onde a coluna 'Ocorrência' contém as palavras-chave 'Prefeitura', 'Prevrio' ou 'período de movimentação', independentemente de maiúsculas e minúsculas. Os resultados serão consolidados em um novo DataFrame.

In [18]:
filtered_data_ocorrencia = pd.DataFrame()
keywords = ['prefeitura', 'prevrio', 'período de movimentação']

print("Filtrando a coluna 'Ocorrência' em cada aba pelos termos:", keywords)

for sheet_name, df in excel_data.items():
    if 'Ocorrência' in df.columns:
        # Converte a coluna 'Ocorrência' para string e aplica o filtro case-insensitive
        # Usa '(?i)' para case-insensitive e '|' para 'ou' entre as palavras-chave
        pattern = '|'.join(keywords)
        df_filtered_sheet = df[df['Ocorrência'].astype(str).str.contains(pattern, case=False, na=False)]

        if not df_filtered_sheet.empty:
            # Adiciona uma coluna 'Aba Original' antes de concatenar
            df_filtered_sheet = df_filtered_sheet.copy() # Evita SettingWithCopyWarning
            df_filtered_sheet['Aba Original'] = sheet_name
            filtered_data_ocorrencia = pd.concat([filtered_data_ocorrencia, df_filtered_sheet], ignore_index=True)
            print(f"  - Aba '{sheet_name}': Encontrado {len(df_filtered_sheet)} registros.")
        else:
            print(f"  - Aba '{sheet_name}': Nenhum registro encontrado com as palavras-chave na coluna 'Ocorrência'.")
    else:
        print(f"  - Aba '{sheet_name}': Coluna 'Ocorrência' não encontrada.")

print(f"\nTotal de registros encontrados em todas as abas com as palavras-chave na coluna 'Ocorrência': {len(filtered_data_ocorrencia)}.")

if not filtered_data_ocorrencia.empty:
    print("Primeiras 5 linhas do DataFrame filtrado por Ocorrência:")
    display(filtered_data_ocorrencia.head())
else:
    print("Nenhum registro foi encontrado após o filtro na coluna 'Ocorrência'.")

Filtrando a coluna 'Ocorrência' em cada aba pelos termos: ['prefeitura', 'prevrio', 'período de movimentação']
  - Aba '10': Encontrado 825 registros.
  - Aba '558': Encontrado 276 registros.
  - Aba '1936': Encontrado 23 registros.

Total de registros encontrados em todas as abas com as palavras-chave na coluna 'Ocorrência': 1124.
Primeiras 5 linhas do DataFrame filtrado por Ocorrência:


,Tipo Atendimento,No.Atendimento,Operador,Data de entrada,Matrícula,Nome,Plano,Telefone,Endereço,Bairro,...,motivo,subMotivo,TipoPlano,Possui anexo,Urgencia,Empresa,Nome da Empresa,Data de inicio do Contrato,Situação Contrato,Aba Original
0,10-INFORMACAO_CONTRATUAL,30922220260701027515,SOARESAL,2026-07-01 09:54:14,004653.0005158.0,PAULO CESAR BORGES DE OLIVEIRA,737CP - RIOSERV MAX QC,21 34267348,RUA SAO MARCELINO 170 CASA,CAMPO GRANDE,...,NaN,NaN,BASICO,NaN,NaN,4653,PREFEITURA DA CIDADE DO RIO DE JANEIRO,42156.0,Ativo,10
1,10-INFORMACAO_CONTRATUAL,30922220260701030844,MARCOST,2026-07-01 10:06:21,004653.0064476.0,IVAN RAMOS DE MENEZES,737CP - RIOSERV MAX QC,21 24182803/(21) 3551-5687,ESTRADA GENERAL PESSOA CAVALCANTI 497 LARGO DO...,GUARATIBA,...,NaN,NaN,BASICO,NaN,NaN,4653,PREFEITURA DA CIDADE DO RIO DE JANEIRO,42156.0,Ativo,10
2,10-INFORMACAO_CONTRATUAL,30922220260701065931,CORREIA,2026-07-01 13:39:01,004653.0120596.0,MARIA DE FATIMA MATTOS,737CP - RIOSERV MAX QC,(21)3971-1818,RUA SEIS 731 CASA,MAUA/MAGE,...,NaN,NaN,BASICO,NaN,NaN,4653,PREFEITURA DA CIDADE DO RIO DE JANEIRO,42156.0,Ativo,10
3,10-INFORMACAO_CONTRATUAL,30922220260702029311,MARCOST,2026-07-02 10:15:43,004653.0099532.0,VALERIA GOES CORREA,737CP - RIOSERV MAX QC,(21)9958-8008,RUA JOSE HIGINO 58 APTO 206,TIJUCA,...,NaN,NaN,BASICO,NaN,NaN,4653,PREFEITURA DA CIDADE DO RIO DE JANEIRO,42156.0,Ativo,10
4,10-INFORMACAO_CONTRATUAL,30922220260702034538,WAGNER,2026-07-02 10:43:48,004653.0089686.0,LEILA ALVES,737CP - RIOSERV MAX QC,21 32733584,RUA MAGALHAES COUTO 758 BL.2 APTO.204,MEIER,...,NaN,NaN,BASICO,NaN,NaN,4653,PREFEITURA DA CIDADE DO RIO DE JANEIRO,42156.0,Ativo,10


### Quantidade Mensal de Ocorrências Filtradas

Vamos agora analisar a distribuição mensal dos registros encontrados na coluna 'Ocorrência' que contêm as palavras-chave 'Prefeitura', 'Prevrio' ou 'período de movimentação'.

In [21]:
if not filtered_data_ocorrencia.empty:
    # Garante que 'Data de entrada' é do tipo datetime
    filtered_data_ocorrencia['Data de entrada'] = pd.to_datetime(filtered_data_ocorrencia['Data de entrada'], errors='coerce')

    # Remove linhas com 'Data de entrada' inválida após a conversão
    df_ocorrencia_cleaned = filtered_data_ocorrencia.dropna(subset=['Data de entrada'])

    if not df_ocorrencia_cleaned.empty:
        # Extrai o ano e o mês
        df_ocorrencia_cleaned['AnoMes'] = df_ocorrencia_cleaned['Data de entrada'].dt.to_period('M')

        # Conta a quantidade por mês para as ocorrências filtradas
        monthly_counts_ocorrencia = df_ocorrencia_cleaned['AnoMes'].value_counts().sort_index().reset_index()
        monthly_counts_ocorrencia.columns = ['AnoMes', 'Quantidade']

        print("\n--- Quantidade de entradas por mês para as Ocorrências Filtradas ---")
        print(monthly_counts_ocorrencia.to_string(index=False))
    else:
        print("Nenhuma 'Data de entrada' válida encontrada no DataFrame filtrado por ocorrência para análise mensal.")
else:
    print("O DataFrame filtrado por ocorrência está vazio. Não é possível calcular a quantidade por mês.")


--- Quantidade de entradas por mês para as Ocorrências Filtradas ---
 AnoMes  Quantidade
2026-07         338
2026-08         601
2026-09         185


### Colunas por Aba Original no DataFrame Filtrado

A pedido, segue novamente a listagem das colunas presentes em cada 'Aba Original' dentro do DataFrame `filtered_data_with_sheets`, que contém as matrículas filtradas e a identificação de sua aba de origem.

In [22]:
if not filtered_data_with_sheets.empty:
    print("\n--- Colunas para cada Aba Original no DataFrame filtrado ---")
    for original_sheet in filtered_data_with_sheets['Aba Original'].unique():
        df_subset = filtered_data_with_sheets[filtered_data_with_sheets['Aba Original'] == original_sheet]
        print(f"Aba Original '{original_sheet}':")
        print(df_subset.columns.tolist())
        print("\n")
else:
    print("O DataFrame filtrado está vazio. Não é possível mostrar as colunas por aba.")


--- Colunas para cada Aba Original no DataFrame filtrado ---
Aba Original '10':
['Tipo Atendimento', 'No.Atendimento', 'Operador', 'Data de entrada', 'Matrícula', 'Nome', 'Plano', 'Telefone', 'Endereço', 'Bairro', 'Cidade', 'Status', 'Data de fechamento', 'Motivo', 'Ocorrência', 'Providências', 'Pjotinha', 'Senha', 'Código do Solicitante', 'Solicitante', 'CRM', 'Código do Grupo', 'Grupo', 'Especialidade', 'Código do Executor', 'Executor', 'Código do Grupo.1', 'Grupo.1', 'CID', 'Diagnóstico', 'TUSS Inicial', 'Operador Final', 'PRC', 'Tempo de Assim', 'Idade', 'motivo', 'subMotivo', 'TipoPlano', 'Possui anexo', 'Urgencia', 'Empresa', 'Nome da Empresa', 'Data de inicio do Contrato', 'Situação Contrato', 'Aba Original']


Aba Original '558':
['Tipo Atendimento', 'No.Atendimento', 'Operador', 'Data de entrada', 'Matrícula', 'Nome', 'Plano', 'Telefone', 'Endereço', 'Bairro', 'Cidade', 'Status', 'Data de fechamento', 'Motivo', 'Ocorrência', 'Providências', 'Pjotinha', 'Senha', 'Código do Sol

### Quantidade Mensal por Aba Original para Ocorrências Filtradas

Agora, vamos analisar a distribuição mensal dos registros filtrados pela coluna 'Ocorrência', separados pela 'Aba Original' de onde vieram.

In [23]:
if not filtered_data_ocorrencia.empty:
    # Garante que 'Data de entrada' é do tipo datetime
    filtered_data_ocorrencia['Data de entrada'] = pd.to_datetime(filtered_data_ocorrencia['Data de entrada'], errors='coerce')

    # Remove linhas com 'Data de entrada' inválida após a conversão
    df_ocorrencia_cleaned_by_sheet = filtered_data_ocorrencia.dropna(subset=['Data de entrada'])

    if not df_ocorrencia_cleaned_by_sheet.empty:
        # Extrai o ano e o mês
        df_ocorrencia_cleaned_by_sheet['AnoMes'] = df_ocorrencia_cleaned_by_sheet['Data de entrada'].dt.to_period('M')

        # Conta a quantidade por mês para cada 'Aba Original'
        monthly_counts_ocorrencia_by_sheet = df_ocorrencia_cleaned_by_sheet.groupby(['AnoMes', 'Aba Original']).size().unstack(fill_value=0)

        # Calcula o total mensal
        monthly_counts_ocorrencia_by_sheet['Total'] = monthly_counts_ocorrencia_by_sheet.sum(axis=1)

        print("\n--- Quantidade de entradas por mês por Aba Original para Ocorrências Filtradas ---")
        print(monthly_counts_ocorrencia_by_sheet.to_string())
    else:
        print("Nenhuma 'Data de entrada' válida encontrada no DataFrame filtrado por ocorrência e por aba para análise mensal.")
else:
    print("O DataFrame filtrado por ocorrência está vazio. Não é possível calcular a quantidade por mês e por aba.")


--- Quantidade de entradas por mês por Aba Original para Ocorrências Filtradas ---
Aba Original   10  1936  558  Total
AnoMes                             
2026-07       234    11   93    338
2026-08       449    12  140    601
2026-09       142     0   43    185


### Exportar Resultados para Excel

Vamos agora consolidar todas as análises realizadas em um único arquivo Excel, com cada conjunto de dados em uma aba separada para facilitar a visualização e o compartilhamento.

In [24]:
output_excel_path = '/content/resultados_analise.xlsx'

with pd.ExcelWriter(output_excel_path, engine='xlsxwriter') as writer:
    # Escreve o DataFrame de contagens mensais de todas as abas
    if 'monthly_counts_all_sheets' in locals() and not monthly_counts_all_sheets.empty:
        monthly_counts_all_sheets.to_excel(writer, sheet_name='Mensal Total', index=False)
        print("\n'Contagens Mensais Totais' exportadas para a aba 'Mensal Total'.")
    else:
        print("\n'Contagens Mensais Totais' está vazio ou não definido, não será exportado.")

    # Escreve o DataFrame de contagens mensais filtradas por matrícula
    if 'monthly_counts_filtered' in locals() and not monthly_counts_filtered.empty:
        monthly_counts_filtered.to_excel(writer, sheet_name='Mensal Matrícula Filtrada', index=False)
        print("'Contagens Mensais por Matrícula Filtrada' exportadas para a aba 'Mensal Matrícula Filtrada'.")
    else:
        print("'Contagens Mensais por Matrícula Filtrada' está vazio ou não definido, não será exportado.")

    # Escreve o DataFrame de contagens mensais filtradas por matrícula e por aba
    if 'monthly_counts_by_sheet' in locals() and not monthly_counts_by_sheet.empty:
        monthly_counts_by_sheet.to_excel(writer, sheet_name='Mensal Matrícula por Aba')
        print("'Contagens Mensais por Matrícula e por Aba' exportadas para a aba 'Mensal Matrícula por Aba'.")
    else:
        print("'Contagens Mensais por Matrícula e por Aba' está vazio ou não definido, não será exportado.")

    # Escreve o DataFrame de dados filtrados pela coluna 'Ocorrência'
    if 'filtered_data_ocorrencia' in locals() and not filtered_data_ocorrencia.empty:
        filtered_data_ocorrencia.to_excel(writer, sheet_name='Filtrado Ocorrência', index=False)
        print("'Dados Filtrados por Ocorrência' exportados para a aba 'Filtrado Ocorrência'.")
    else:
        print("'Dados Filtrados por Ocorrência' está vazio ou não definido, não será exportado.")

    # Escreve o DataFrame de contagens mensais de ocorrências por aba
    if 'monthly_counts_ocorrencia_by_sheet' in locals() and not monthly_counts_ocorrencia_by_sheet.empty:
        monthly_counts_ocorrencia_by_sheet.to_excel(writer, sheet_name='Mensal Ocorrência por Aba')
        print("'Contagens Mensais de Ocorrência por Aba' exportadas para a aba 'Mensal Ocorrência por Aba'.")
    else:
        print("'Contagens Mensais de Ocorrência por Aba' está vazio ou não definido, não será exportado.")

    # Escreve o DataFrame completo filtrado por matrícula com a aba original
    if 'filtered_data_with_sheets' in locals() and not filtered_data_with_sheets.empty:
        filtered_data_with_sheets.to_excel(writer, sheet_name='Dados Matrícula por Aba', index=False)
        print("'Dados Filtrados por Matrícula com Aba Original' exportados para a aba 'Dados Matrícula por Aba'.")
    else:
        print("'Dados Filtrados por Matrícula com Aba Original' está vazio ou não definido, não será exportado.")

print(f"\nTodos os resultados foram exportados para o arquivo Excel: {output_excel_path}")


'Contagens Mensais Totais' exportadas para a aba 'Mensal Total'.
'Contagens Mensais por Matrícula Filtrada' exportadas para a aba 'Mensal Matrícula Filtrada'.
'Contagens Mensais por Matrícula e por Aba' exportadas para a aba 'Mensal Matrícula por Aba'.
'Dados Filtrados por Ocorrência' exportados para a aba 'Filtrado Ocorrência'.
'Contagens Mensais de Ocorrência por Aba' exportadas para a aba 'Mensal Ocorrência por Aba'.
'Dados Filtrados por Matrícula com Aba Original' exportados para a aba 'Dados Matrícula por Aba'.

Todos os resultados foram exportados para o arquivo Excel: /content/resultados_analise.xlsx
